# Decision Trees:

---

## Importing Libraries:

In [1]:
import time

import numpy as np

import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, make_scorer

## Importing Dataset:

In [2]:
data = pd.read_csv("email_spam_features.csv")

## Building Base Model:

In [3]:
X = data.drop(columns=['Category'])
y = data['Category']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=69)

In [4]:
label = LabelEncoder()

y_train = label.fit_transform(y_train)
y_test = label.transform(y_test)

In [5]:
dct = DecisionTreeClassifier()

start_time = time.time()
dct.fit(X_train, y_train)
end_time = time.time() - start_time

print('Model Trained Sucessfully!')
print(f'Time Taken For Training: {end_time:.2f}s')

Model Trained Sucessfully!
Time Taken For Training: 0.04s


In [6]:
y_pred = dct.predict(X_test)

print(f"Accuracy: {(accuracy_score(y_test, y_pred) * 100):.2f}%")

print(f"\nPrecision: {(precision_score(y_test, y_pred)*100):.2f}%")

print(f"\nRecall: {(recall_score(y_test, y_pred)*100):.2f}%")

print(f"\nF1 Score: {(f1_score(y_test, y_pred)*100):.2f}%")

Accuracy: 97.67%

Precision: 93.91%

Recall: 86.40%

F1 Score: 90.00%


In [7]:
matrix = confusion_matrix(y_test, y_pred)
matrix_df = pd.DataFrame(matrix, index=['Actual ham', 'Actual spam'], columns=['Predicted ham', 'Predicted spam'])
matrix_df

,Predicted ham,Predicted spam
Actual ham,900,7
Actual spam,17,108


## Hyper-Parameter Tuning:

In [8]:
params = {'criterion' : ['gini', 'entropy', 'log_loss'],
          'splitter' : ['best', 'random'],
          'max_depth' : [3, 4, 5, 6, 7, 8, 9, 10, None],
          'max_features' : ['sqrt', 'log2', None],
          'class_weight' : ['balanced', None]}

grid = GridSearchCV(
    estimator=DecisionTreeClassifier(),
    param_grid=params,
    scoring='precision',
    cv=5,
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)

Best Parameters: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 3, 'max_features': 'sqrt', 'splitter': 'random'}


In [9]:
params = {'criterion' : ['gini', 'entropy', 'log_loss'],
          'splitter' : ['best', 'random'],
          'max_depth' : [3, 4, 5, 6, 7, 8, 9, 10, None],
          'max_features' : ['sqrt', 'log2', None],
          'class_weight' : ['balanced', None]}

grid = GridSearchCV(
    estimator=DecisionTreeClassifier(),
    param_grid=params,
    scoring='f1',
    cv=5,
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)

Best Parameters: {'class_weight': None, 'criterion': 'gini', 'max_depth': 8, 'max_features': None, 'splitter': 'random'}


In [10]:
# Priority -> Precision:

Dct = DecisionTreeClassifier(criterion='entropy', splitter='random', max_depth=3, max_features='sqrt', class_weight=None)

start_time = time.time()
Dct.fit(X_train, y_train)
end_time = time.time() - start_time

print('Model Trained Sucessfully!')
print(f'Time Taken For Training: {end_time:.2f}s')

Model Trained Sucessfully!
Time Taken For Training: 0.01s


In [11]:
y_pred = Dct.predict(X_test)

print(f"Accuracy: {(accuracy_score(y_test, y_pred) * 100):.2f}%")

print(f"\nPrecision: {(precision_score(y_test, y_pred)*100):.2f}%")

print(f"\nRecall: {(recall_score(y_test, y_pred)*100):.2f}%")

print(f"\nF1 Score: {(f1_score(y_test, y_pred)*100):.2f}%")

Accuracy: 94.57%

Precision: 74.47%

Recall: 84.00%

F1 Score: 78.95%


In [12]:
matrix = confusion_matrix(y_test, y_pred)
matrix_df = pd.DataFrame(matrix, index=['Actual ham', 'Actual spam'], columns=['Predicted ham', 'Predicted spam'])
matrix_df

,Predicted ham,Predicted spam
Actual ham,871,36
Actual spam,20,105


In [13]:
# Priority -> F1:

DCT = DecisionTreeClassifier(criterion='gini', splitter='random', max_depth=8, max_features=None, class_weight=None)

start_time = time.time()
DCT.fit(X_train, y_train)
end_time = time.time() - start_time

print('Model Trained Sucessfully!')
print(f'Time Taken For Training: {end_time:.2f}s')

Model Trained Sucessfully!
Time Taken For Training: 0.01s


In [14]:
y_pred = DCT.predict(X_test)

print(f"Accuracy: {(accuracy_score(y_test, y_pred) * 100):.2f}%")

print(f"\nPrecision: {(precision_score(y_test, y_pred)*100):.2f}%")

print(f"\nRecall: {(recall_score(y_test, y_pred)*100):.2f}%")

print(f"\nF1 Score: {(f1_score(y_test, y_pred)*100):.2f}%")

Accuracy: 97.48%

Precision: 91.60%

Recall: 87.20%

F1 Score: 89.34%


In [15]:
matrix = confusion_matrix(y_test, y_pred)
matrix_df = pd.DataFrame(matrix, index=['Actual ham', 'Actual spam'], columns=['Predicted ham', 'Predicted spam'])
matrix_df

,Predicted ham,Predicted spam
Actual ham,897,10
Actual spam,16,109


## Cross-Validation:

In [16]:
# Priority -> Precision:

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=69)

scoring = {
    'accuracy' : 'accuracy',
    'precision' : make_scorer(precision_score, pos_label='spam'),
    'recall' : make_scorer(recall_score, pos_label='spam'),
    'f1' : make_scorer(f1_score, pos_label='spam')
}

result = cross_validate(
    estimator=DecisionTreeClassifier(criterion='entropy', splitter='random', max_depth=3, max_features='sqrt', class_weight=None),
    X=X,
    y=y,
    scoring=scoring,
    cv=cv,
    n_jobs=-1
)

metrics = ['accuracy', 'precision', 'recall', 'f1']

for i in metrics:
    score = result[f'test_{i}']
    print(f"{i.capitalize()}: {np.mean(score)*100:.2f}%\n")

Accuracy: 93.41%

Precision: 97.10%

Recall: 49.14%

F1: 61.25%



In [17]:
# Priority -> F1:

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=69)

scoring = {
    'accuracy' : 'accuracy',
    'precision' : make_scorer(precision_score, pos_label='spam'),
    'recall' : make_scorer(recall_score, pos_label='spam'),
    'f1' : make_scorer(f1_score, pos_label='spam')
}

result = cross_validate(
    estimator=DecisionTreeClassifier(criterion='gini', splitter='random', max_depth=8, max_features=None, class_weight=None),
    X=X,
    y=y,
    scoring=scoring,
    cv=cv,
    n_jobs=-1
)

metrics = ['accuracy', 'precision', 'recall', 'f1']

for i in metrics:
    score = result[f'test_{i}']
    print(f"{i.capitalize()}: {np.mean(score)*100:.2f}%\n")

Accuracy: 98.06%

Precision: 95.37%

Recall: 88.77%

F1: 91.91%

